In [5]:
import random, warnings
from collections import defaultdict, Counter
from fileinput import filename

import numpy as np
import pandas as pd
import networkx as nx
from itertools import combinations
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ndlib epidemic models
import ndlib.models.epidemics as ep
import ndlib.models.ModelConfig as mc

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

# Dataset paths
DATASETS = {
    "PROPERTY_PROPOSAL (LARGE)":  "PROPERTY_PROPOSAL.csv",
    "PROPERTIES (MEDIUM)":        "PROPERTIES.csv",
    "REQUEST_A_QUERY (SMALL)":    "REQUEST_A_QUERY.csv",
}

#Defining the network

In [6]:
def building_network(filename, max_thread_size =200):
    #load data
    df = pd.read_csv(filename,skipinitialspace=True)
    #a social 'event' = one thread on one page
    thread_users = defaultdict(set)
    for _, row in df.iterrows():
        key = (row['page_name'],row['thread_subject'])
        thread_users[key].add(row['username'])
    #user activity are the total comments per user
    user_activity = df.groupby('username').size().to_dict()
    #enumerate co-comment pairs --> weighted edges
    #for each thread take all c(k,2)user pairs.
    #shared-event counts --> edge weight accumulated across threads
    edge_weights = defaultdict(int)
    skipped = 0
    for users_set in thread_users.values():
        ul = sorted(users_set)
        if len(ul) > max_thread_size:
            skipped += 1
            continue
        for i in range(len(ul)):
            for j in range (i + 1,len(ul)):
                edge_weights[(ul[i],ul[j])] += 1
    # assemble the graph
    graph = nx.Graph()
    for user, act in user_activity.items():
        graph.add_node(user, activity=act)
    for (u,v), w in edge_weights.items():
        graph.add_edge(u,v, weight=w)

    giant = max(nx.connected_components(graph), key = len) #Finding the largest connected subgraph
    graph_giant = graph.subgraph(giant).copy() #small disconnected groups are not useful for propogation analysis

    return graph,graph_giant,df

#TASK A - NETWORK CONSTRUCTION

In [7]:
def task_a(datasets):
    graphs = {} #dictionary to store samples
    for label, filename in datasets.items():
        print(f"  Dataset: {label}\n")
        graph, graph_giant, df = building_network(filename)
        graphs[label] = (graph,graph_giant,df) #saves graphs for later use

    #Fig1: spring layout visualization for all 3 networks
    palette = ['#2196F3', '#FF9800', '#4CAF50']
    titles = [
        "PROPERTY_PROPOSAL\n(LARGE — 150-node sample)",
        "PROPERTIES\n(MEDIUM — 150-node sample)",
        "REQUEST_A_QUERY\n(SMALL — full giant component)",
    ]
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle(
        "Task A — Wikidata Editor Co-Comment Networks\n",
        fontsize=13, fontweight='bold'
    )
    for ax, (label, (graph, graph_giant, df)), color, title in zip(axes, graphs.items(), palette, titles): # iterates through subplot axes, graphs, colors and titles
        # sample 150 nodes for large graphs
        sample_nodes = list(graph_giant.nodes())
        if len(sample_nodes)>150: #if graph has more than 150 nodes, sampling is used
            top_nodes = [n for n, _ in
                         sorted(graph_giant.degree(), key=lambda x: x[1], reverse=True)[:50]] #selects 50 most connected editors as influential nodes
            rest = [n for n in sample_nodes if n not in top_nodes] #adds random nodes to make total around 150 nodes
            sample_nodes = top_nodes + random.sample(rest, min(100, len(rest)))

        graph_sub = graph_giant.subgraph(sample_nodes) #creates subgraph - smaller network containing only sampled nodes
        pos  = nx.spring_layout(graph_sub, seed=42, k=1.2 / len(graph_sub)**0.5) #use force-directed layout - common for social networks
        degs = dict(graph_sub.degree())
        node_sz = [10 + degs[n] * 4 for n in graph_sub.nodes()] #nodes with more connections become larger

        nx.draw_networkx_nodes(graph_sub, pos, ax=ax, node_size=node_sz,
                               node_color=color, alpha=0.75) #draw nodes
        nx.draw_networkx_edges(graph_sub, pos, ax=ax, alpha=0.15,
                               edge_color='grey', width=0.5) #draw edges
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.axis('off')

    plt.tight_layout()
    plt.savefig("Task A Wikidata Editor Co-Comment Networks.png", dpi=130)
    plt.show()

    #Figure 2: degree distributions
    fig2, axes2 = plt.subplots(1,3, figsize= (15,4)) #each plot show degree distribution
    fig2.suptitle ("TASK A - Degree Distributions (log-log scale)", fontsize=10, fontweight='bold')

    for ax,(label,(graph,graph_giant,df)), color in zip(axes2, graphs.items(), palette):
        degs = sorted([d for _, d in graph.degree() if d > 0]) #get all the node degrees
        cnt = Counter(degs) #count frequency ex: degree 1 --> 40 nodes
        #Convert to arrays
        xs = np.array(sorted(cnt.keys()), float)
        ys = np.array([cnt[int(x)] for x in xs], float)

        #Plot Log-log distribution as log-log scale helps detect power-law behaviour
        ax.loglog(xs, ys, 'o', color=color, markersize=5, markeredgecolor='k', alpha=0.8)
        #fit power-law slope in log-log space
        slope, intercept = np.polyfit(np.log(xs), np.log(ys), 1) #fits linear model in log space. Slope indicates power-law exponent
        fit_y = np.exp(intercept + slope * np.log(xs)) #Shows best fit line
        ax.loglog(xs, fit_y, '--', color='black', label = f's slope = {slope:.2f}')

        ax.set_title(label.split(' (')[0].strip(), fontsize=10, fontweight='bold')
        ax.set_xlabel("Degree k")
        ax.set_ylabel("Count")

    plt.tight_layout()
    plt.savefig("Task A Degree Distributions.png", dpi=130)

    return graphs

#TASK B - NETWORK METRICS AND SMALL-WORLD ANALYSIS

In [8]:
def task_b(graphs):
    print("  TASK B — NETWORK METRICS & SMALL-WORLD ANALYSIS")
    results ={}
    palette = ['#2196F3', '#FF9800', '#4CAF50']

    for label,(graph, graph_giant, df) in graphs.items(): #processes for each dataset seperately
        print(f"  Dataset: {label}\n")

        n = graph_giant.number_of_nodes() #number of editors
        m = graph_giant.number_of_edges() #number of interactions
        avg_k = 2 * m / n #average number of connections
        density = nx.density(graph_giant) #how connected the network is

        #CLustering coefficient
        avg_C = nx.average_clustering(graph_giant) #measuring local clustering

        #Case 1 - large networks
        #Average shortest path nd diameter (Sampling from large graph for speed)
        if n > 500:
            sources = random.sample(list(graph_giant.nodes()), 350) #pick 350 random nodes
            lengths = []
            for src in sources:
                spl = nx.single_source_shortest_path_length(graph_giant, src) #Computing the shortest path
                lengths.extend(spl.values())
            avg_L = float(np.mean(lengths)) #collect all path lengths and find the mean
            diam = max(
                max(nx.single_source_shortest_path_length(graph_giant, s).values())
                for s in random.sample(list(graph_giant.nodes()), 50) #diameter is the longest shortest path, sampling to reduce computations
            )
        else:  #if network is small, compute exact values
            avg_L =nx.average_shortest_path_length(graph_giant)
            diam = nx.diameter(graph_giant)

        #Random network baseline
        #Erdos-renyi baseline with same n, p = density)
        #code compares real network to a random network
        #for ER: expected clustering = p =density, L_rand = ln(n) / ln(<k>)
        C_rand = density
        L_rand = np.log(n) / np.log(avg_k) if avg_k > 1 else float('inf') #random path length estimate, approximating average path lengths in random graphs

        #Small world ratios
        C_ratio = avg_C / C_rand #C_ratio >>1 --> clustering much higher than random
        L_ratio = avg_L / L_rand  #L_ratio = 1 --> path length similar to random

        # Watts-Strogatz small-world sigma: SW if C >> C_rand AND L ≈ L_rand
        #Based on small world definition
        #sigma > 1 --> small-world
        #sigma = 1 --> random
        #sigma <1 --> not small-world
        sigma   = C_ratio / L_ratio

        #Degree distribution power-law slope
        degs = [d for _, d in graph_giant.degree()]
        cnt = Counter(degs) #Counts how many nodes have each degree
        xs = np.array(sorted([k for k in cnt if k>0]), float)
        ys = np.array([cnt[int(x)] for x in xs], float)
        #slope indicates scale -free behaviour i.e. degree distribution follows power-law
        # ≈ -2 or -3 --> scale-free
        # near 0 --> random
        slope, _ = np.polyfit(np.log(xs), np.log(ys), 1)

        #Identify hub nodes
        #Find top 5 highest-degree nodes i.e. most  influential editors
        top5 = sorted(graph_giant.degree(), key=lambda x: x[1], reverse=True)[:5]

        print(f"  n={n:,}  m={m:,}  <k>={avg_k:.2f}  density={density:.5f}")
        print(f"  C={avg_C:.4f}  L={avg_L:.4f}  diameter={diam}")
        print(f"  C_rand={C_rand:.5f}  L_rand={L_rand:.4f}")
        print(f"  C/C_rand={C_ratio:.2f}  L/L_rand={L_ratio:.2f}  σ={sigma:.2f}")
        print(f"  Degree dist slope={slope:.2f}  (< −1 → scale-free tendency)")
        print(f"  Top-5 hubs: {[(u[:20], d) for u, d in top5]}")


        #Storing the results
        results[label] = dict(
            n = n, m = m, avg_k = avg_k, density = density,
            avg_C = avg_C, avg_L = avg_L, diam  = diam,
            C_rand = C_rand, L_rand = L_rand,
            C_ratio = C_ratio, L_ratio = L_ratio,sigma = sigma,
            slope = slope, degs = degs, top5=top5
        )
    # Metric comparison
    fig, axes = plt.subplots(2,3,figsize =(16,9))
    fig.suptitle("Task B - Network Metric Analysis")
    net_names = ['PROP_PROPOSAL\n(LARGE)', 'PROPERTIES\n(MEDIUM)',
                 'REQ_QUERY\n(SMALL)']
    x = np.arange(3)
    w= 0.30

    #Row 0
    #Degree distribution plots per network(log-log)
    for ax, (label, res), color, name in zip(
        axes[0], results.items(), palette, net_names):
        cnt = Counter(res['degs'])
        xs  = np.array(sorted([k for k in cnt if k > 0]), float)
        ys  = np.array([cnt[int(k)] for k in xs], float)
        ax.loglog(xs, ys, 'o', color=color, markersize=3.5, alpha=0.75) #log-log plot to show if the network follows power-law distribution
        fit_y = np.exp(np.mean(np.log(ys)) +
                       res['slope'] * (np.log(xs) - np.mean(np.log(xs)))) #Fitted line plotted
        ax.loglog(xs, fit_y, '--', color='black', lw=1.3,
                  label=f"slope={res['slope']:.2f}")
        ax.set_title(f"{name}\nDegree distribution", fontsize=9)
        ax.set_xlabel("Degree k"); ax.set_ylabel("Count")
        ax.legend(fontsize=8)

    # Row 1 col 0
    # clustering actual vs random
    #If actual >> random --> strong community structure
    ax = axes[1][0]
    C_actual = [r['avg_C']  for r in results.values()]
    C_rands  = [r['C_rand'] for r in results.values()]
    b1 = ax.bar(x - w/2, C_actual, w, color=palette, alpha=0.85, label='Actual C')
    ax.bar(x + w/2, C_rands,  w, color='lightgrey', edgecolor='black', label='ER C')
    ax.set_xticks(x); ax.set_xticklabels(['LARGE', 'MEDIUM', 'SMALL'], fontsize=9)
    ax.set_ylabel("Clustering coefficient C")
    ax.set_title("Clustering: Actual vs ER Random")
    ax.legend(fontsize=8)
    for bar, v in zip(b1, C_actual):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{v:.3f}', ha='center', fontsize=8)

    # Row 1 col 1
    #path length actual vs random
    #Small world networks typically have similar path lengths to random graphs
    ax = axes[1][1]
    L_actual = [r['avg_L']  for r in results.values()]
    L_rands  = [r['L_rand'] for r in results.values()]
    b2 = ax.bar(x - w/2, L_actual, w, color=palette, alpha=0.85, label='Actual L')
    ax.bar(x + w/2, L_rands, w, color='lightgrey', edgecolor='black', label='ER L')
    ax.set_xticks(x); ax.set_xticklabels(['LARGE', 'MEDIUM', 'SMALL'], fontsize=9)
    ax.set_ylabel("Avg shortest path length L")
    ax.set_title("Path Length: Actual vs ER Random")
    ax.legend(fontsize=8)
    for bar, v in zip(b2, L_actual):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                f'{v:.2f}', ha='center', fontsize=8)

    # Row 1 col 2
    # small-world sigma
    #bar chart of sigma values
    # sigma = 1 --> random network
    #If bars are above the line, the network is small world
    ax = axes[1][2]
    sigmas = [r['sigma'] for r in results.values()]
    b3 = ax.bar(['LARGE', 'MEDIUM', 'SMALL'], sigmas,
                color=palette, alpha=0.85, edgecolor='black')
    ax.axhline(1, color='red', linestyle='--', lw=1.5, label='σ=1 (random)')
    ax.set_ylabel("Small-world sigma σ")
    ax.set_title("Small-World Index σ = (C/C_rand) / (L/L_rand)")
    ax.legend(fontsize=8)
    for bar, v in zip(b3, sigmas):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{v:.1f}', ha='center', fontsize=9, fontweight='bold')

    plt.tight_layout()
    plt.savefig("task_b_metrics.png", dpi=130, bbox_inches='tight')

    # Summary table

    print("TASK B — SUMMARY TABLE")

    print(f"{'Metric':<28} {'PROP_PROPOSAL':>14} {'PROPERTIES':>14} {'REQ_QUERY':>14}")

    rows_display = [
        ('n',       'Nodes (giant)'),
        ('m',       'Edges'),
        ('avg_k',   'Avg degree <k>'),
        ('density', 'Density'),
        ('avg_C',   'Clustering C'),
        ('avg_L',   'Avg path L'),
        ('diam',    'Diameter'),
        ('C_rand',  'C_random (ER)'),
        ('L_rand',  'L_random (ER)'),
        ('C_ratio', 'C / C_rand'),
        ('L_ratio', 'L / L_rand'),
        ('sigma',   'σ (small-world)'),
        ('slope',   'Deg dist slope'),
    ]
    vals = list(results.values())
    for key, name in rows_display:
        row = [f"{r[key]:.4f}" if isinstance(r[key], float) else str(r[key])
               for r in vals]
        print(f"  {name:<26} {row[0]:>14} {row[1]:>14} {row[2]:>14}")

    return results





TASK B - NETWORK METRICS

#TASK C- EPIDEMIC/TROLL PROPOGATION MODEL

#Simple discrete-time SIR epidemic on graph G.

#SIR states
    S --> susceptible (normal editor)
    I --> Infected (trolling editor)
    R --> Recovered (moderator stopped them/no longer trolling)

#Parameters
    graph  : networkx.Graph
    seeds  : list of initially Infected nodes
    beta   : per-edge infection probability per step
    gamma  : per-node recovery probability per step
    steps  : number of time steps to simulate

#Returns (S_t, I_t, R_t) — lists of counts at each step.

In [9]:
warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

#Calculating Editor risk score
#risk combines 3 network metrics --> mean (normalised degree, normalised activity, normalised betweenness)
#Higher = more central / more active = higher troll-spread risk.
def compute_risk(graph):
    deg_dict = dict(graph.degree()) #number of connections an editor has --> more connections means higher spread potential
    act_dict = nx.get_node_attributes(graph, 'activity') #more comments --> more exposure --> higher risk
    bet_dict = nx.betweenness_centrality(
        graph, k=min(200, graph.number_of_nodes()), normalized=True, seed=42
    ) #Measures how often a node lies on the shortest paths --> high betweenness = bridge between communities. These users can spread behaviour across groups

    #Normalisation
    def norm(d):
        mn, mx = min(d.values()), max(d.values())
        if mx == mn:
            return {k: 0.0 for k in d}
        return {k: (v - mn) / (mx - mn) for k, v in d.items()} #(v-min)/(max-min)

    nd, na, nb = norm(deg_dict), norm(act_dict), norm(bet_dict)
    return {u: (nd[u] + na[u] + nb[u]) / 3.0 for u in graph.nodes()} #final risk score


#  ndlib SIR simulation
#  Run a SIR epidemic on graph using ndlib.
#SIR states:
    S = Susceptible
    I = Infected
    R = Recovered
#  Parameters
    # graph          : networkx.Graph - the contact network
    # infected_nodes : initially infected nodes (seeds)
    # beta           : infection probability per edge per step
    # gamma          : recovery probability per step
    # iterations     : number of discrete time steps

# Returns
    #trends : dict returned by ndlib iteration_bunch()


def run_sir_ndlib(graph, infected_nodes, beta=0.15, gamma=0.05, iterations=30):

    # Choose model
    model = ep.SIRModel(graph) #graph becomes contact network

    # configure parameters
    cfg = mc.Configuration()
    cfg.add_model_parameter('beta',  beta)   # S→I infection rate
    cfg.add_model_parameter('gamma', gamma)  # I→R recovery rate -->probability troll stops trolling

    # Seed the initial infected set explicitly.
    # ndlib accepts a list of node ids via 'Infected' in model parameters.
    cfg.add_model_parameter('fraction_infected', 0)
    model.set_initial_status(cfg)

    # Manually set seeds to Infected (status=1); all others default to S (0)
    initial_status = {node: 0 for node in graph.nodes()}   # 0 = Susceptible
    for node in infected_nodes:
        if node in initial_status:
            initial_status[node] = 1                    # 1 = Infected
    model.set_initial_status(cfg)
    # Override with explicit status dict
    model.status = initial_status

    # run for `iterations` steps --> each step returns status changes
    iterations_data = model.iteration_bunch(iterations)

    # unpack trends
    # iteration_bunch returns a list of dicts, each with {'iteration': t, 'status': {node: status_code}}
    # reconstruct S/I/R counts manually for plotting.
    n = graph.number_of_nodes()
    S_counts, I_counts, R_counts = [], [], []

    current_status = initial_status.copy()
    for it in iterations_data:
        # ndlib returns only the *delta* (changed nodes) in 'status'
        current_status.update(it['status'])
        s = sum(1 for v in current_status.values() if v == 0)
        i = sum(1 for v in current_status.values() if v == 1)
        r = sum(1 for v in current_status.values() if v == 2)
        S_counts.append(s)
        I_counts.append(i)
        R_counts.append(r)

    return S_counts, I_counts, R_counts


#  Main logic

def task_c(graphs, b_res):
    print("  TASK C — EPIDEMIC / TROLL PROPAGATION MODEL  (ndlib SIR)")

    all_results = {}
    palette     = ['#2196F3', '#FF9800', '#4CAF50']

    for label, (graph, graph_giant, df) in graphs.items():
        print(f"  Dataset: {label}")

        risk     = compute_risk(graph_giant) # Compute risk
        nx.set_node_attributes(graph_giant, risk, 'risk')
        deg_dict = dict(graph_giant.degree())
        nodes    = list(graph_giant.nodes())

        # Pick 2 random editors (non-isolates)
        active         = [u for u in nodes if deg_dict[u] > 1] # Editors with degree > 1
        editor_A, editor_B = random.sample(active, 2) #These are potential trolls

        print(f"\n  Randomly selected editors:")
        print(f"    A: {editor_A[:50]}  "
              f"deg={deg_dict[editor_A]}  risk={risk[editor_A]:.3f}")
        print(f"    B: {editor_B[:50]}  "
              f"deg={deg_dict[editor_B]}  risk={risk[editor_B]:.3f}")

        # Q1: Shortest path + neighbourhood Jaccard
        # Logic: in a small-world network, a short path means the flagged behaviour is likely already in shared social circles.

        try:
            dist_AB = nx.shortest_path_length(graph_giant, editor_A, editor_B)
        except nx.NetworkXNoPath:
            dist_AB = float('inf')

        neigh_A = set(graph_giant.neighbors(editor_A))
        neigh_B = set(graph_giant.neighbors(editor_B))
        mutual  = neigh_A & neigh_B
        jaccard = (len(mutual) / len(neigh_A | neigh_B) #Measures shared contacts [Intersection/ union] --> High value --> they share community
                   if (neigh_A | neigh_B) else 0.0
                   )

        # Interpretation:
        # distance <= 2 --> high chance
        # distance <= 4 --> moderate
        # distance > 4 --> low

        print(f"\n  Q1 — dist(A,B)={dist_AB}"               f"mutual_neighbours={len(mutual)}  Jaccard={jaccard:.4f}")
        if dist_AB <= 2:
            verdict = "HIGH — direct overlap; spread very plausible already"
        elif dist_AB <= 4:
            verdict = "MODERATE — within small-world radius"
        else:
            verdict = "LOW — editors are far apart in the network"
        print(f"  Spread-already plausibility: {verdict}")

        # Q2a: Only A trolling
        # Priority score = risk(u) / hop_distance(A, u)
        # Rationale: high-risk editors nearby are most likely to have been exposed AND to further propagate the behaviour.
        lengths_A = nx.single_source_shortest_path_length(graph_giant, editor_A, cutoff=4) #Distance from A
        # priority score = risk / distance
        # Meaning high-risk editors + close to A
        priority_one = {
            u: risk[u] / d
            for u, d in lengths_A.items()
            if u != editor_A and d > 0
        }
        top_one = sorted(priority_one.items(),
                         key=lambda x: x[1], reverse=True)[:10] # top 10

        print(f"\n  Q2a — Only A trolling → top 10 to check next:")
        print(f"  {'Rk':<4} {'Username':<38} {'Score':>8}  {'Dist':>5}  {'Deg':>6}")
        for rk, (u, sc) in enumerate(top_one, 1):
            print(f"  {rk:<4} {u[:38]:<38} {sc:>8.4f}  "
                  f"{lengths_A[u]:>5}  {deg_dict[u]:>6}")

        # Q2b: Both trolling
        # Score = risk(u) × (1/dA + 1/dB): reachable from both seeds --> meaning editors from both sources get high priority
        # scores higher — two infection sources converging.
        lengths_B    = nx.single_source_shortest_path_length(graph_giant, editor_B, cutoff=4) # limits BFS to 4 hops
        all_reached  = set(lengths_A) | set(lengths_B)
        priority_both = {
            u: risk[u] * (
                (1 / lengths_A[u] if u in lengths_A and lengths_A[u] > 0 else 0) +
                (1 / lengths_B[u] if u in lengths_B and lengths_B[u] > 0 else 0)
            )
            for u in all_reached if u not in (editor_A, editor_B)
        }
        top_both = sorted(priority_both.items(),
                          key=lambda x: x[1], reverse=True)[:10]

        print(f"\n  Q2b — Both A and B trolling → top 10 to check next:")
        print(f"  {'Rk':<4} {'Username':<38} {'Score':>8}  {'dA':>4}  {'dB':>4}")
        for rk, (u, sc) in enumerate(top_both, 1):
            dA = lengths_A.get(u, '∞')
            dB = lengths_B.get(u, '∞')
            print(f"  {rk:<4} {u[:38]:<38} {sc:>8.4f}  {str(dA):>4}  {str(dB):>4}")

        all_results[label] = dict(
            graph = graph_giant, editor_A=editor_A, editor_B=editor_B,
            dist_AB=dist_AB, jaccard=jaccard, risk=risk,
            deg_dict=deg_dict, lengths_A=lengths_A, lengths_B=lengths_B,
            top_one=top_one, top_both=top_both,
        )

    # ndlib SIR on REQUEST_A_QUERY (smallest graph = fastest)
    print("\n")
    print("  ndlib SIR SIMULATION — REQUEST_A_QUERY (β=0.15, γ=0.05)")


    rq  = all_results["REQUEST_A_QUERY (SMALL)"]
    graph_s = rq['graph']
    eA  = rq['editor_A']
    eB  = rq['editor_B']

    # Case 1: only A infected
    S1, I1, R1 = run_sir_ndlib(graph_s, [eA],      beta=0.15, gamma=0.05)
    # Case 2: both A and B infected
    S2, I2, R2 = run_sir_ndlib(graph_s, [eA, eB],  beta=0.15, gamma=0.05)

    print(f"  Seed=[A only]:  peak infected={max(I1):3d} at step "
          f"{I1.index(max(I1))+1:2d},  total recovered={R1[-1]}")
    print(f"  Seed=[A + B]:   peak infected={max(I2):3d} at step "
          f"{I2.index(max(I2))+1:2d},  total recovered={R2[-1]}")

    # Epidemic visualisation
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle("Task C — Troll Propagation / ndlib SIR Model",
                 fontsize=13, fontweight='bold')

    # Panel 0: SIR curves
    # shows Infected vs time
    # shows Recovered vs time
    ax    = axes[0]
    steps = range(1, len(S1) + 1)
    ax.plot(steps, I1, 'r-o',  markersize=3, label='Infected (seed=A only)')
    ax.plot(steps, R1, 'b--',                label='Recovered (seed=A only)')
    ax.plot(steps, I2, 'r-^',  markersize=3, label='Infected (seed=A+B)')
    ax.plot(steps, R2, 'b:',                 label='Recovered (seed=A+B)')
    ax.set_xlabel("Time step")
    ax.set_ylabel("Number of editors")
    ax.set_title("ndlib SIR Epidemic Curves\n(REQUEST_A_QUERY, β=0.15, γ=0.05)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    # Panel 1: risk-coloured network map
    # nodes colored by risk
    # Green --> low
    # Red --> high
    ax       = axes[1]
    risk_map = rq['risk']
    deg_map  = rq['deg_dict']
    pos      = nx.spring_layout(graph_s, seed=42, k=0.8 / len(graph_s)**0.5)
    node_col = [risk_map[n] for n in graph_s.nodes()]
    node_sz  = [5 + deg_map[n] * 2 for n in graph_s.nodes()]

    nx.draw_networkx_nodes(graph_s, pos, ax=ax, node_size=node_sz,
                           node_color=node_col, cmap='RdYlGn_r',
                           alpha=0.8, vmin=0, vmax=1)
    nx.draw_networkx_edges(graph_s, pos, ax=ax, alpha=0.08, edge_color='grey')

    for editor, marker, col in [(eA, '*', 'red'), (eB, 'D', 'blue')]:
        if editor in pos:
            xy = pos[editor]
            ax.plot(xy[0], xy[1], marker, color=col, markersize=14,
                    markeredgecolor='black', zorder=5)

    ax.set_title("Editor Risk Map (REQUEST_A_QUERY)\n"
                 "Red=high risk  ★=Editor A  ◆=Editor B")
    ax.axis('off')
    plt.colorbar(plt.cm.ScalarMappable(cmap='RdYlGn_r'), ax=ax,
                 label='Risk score', fraction=0.03, pad=0.04)

    # Panel 2: priority bar chart (both trolling scenario)
    #Editors most likely to spread trolling
    ax      = axes[2]
    top_n   = rq['top_both'][:8]
    names   = [u[:22] + '…' if len(u) > 22 else u for u, _ in top_n]
    scores  = [sc for _, sc in top_n]
    y_pos   = range(len(names))
    ax.barh(list(y_pos), scores, color='#E53935', alpha=0.8, edgecolor='black')
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(names, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel("Priority score  (risk × Σ 1/distance)")
    ax.set_title("Q2b — Both Trolling\nEditor Check Priority List")
    ax.grid(alpha=0.3, axis='x')

    plt.tight_layout()
    plt.savefig("task_c_ndlib.png", dpi=130, bbox_inches='tight')

    return all_results

## MAIN

In [10]:
if __name__ == "__main__":
    graphs   = task_a(DATASETS)
    b_res    = task_b(graphs)
    c_res    = task_c(graphs, b_res)
    print("\n\nAll tasks complete. Figures saved to working directory.")

  Dataset: PROPERTY_PROPOSAL (LARGE)

  Dataset: PROPERTIES (MEDIUM)

  Dataset: REQUEST_A_QUERY (SMALL)

  TASK B — NETWORK METRICS & SMALL-WORLD ANALYSIS
  Dataset: PROPERTY_PROPOSAL (LARGE)

  n=3,051  m=46,154  <k>=30.25  density=0.00992
  C=0.8162  L=2.3963  diameter=5
  C_rand=0.00992  L_rand=2.3531
  C/C_rand=82.29  L/L_rand=1.02  σ=80.80
  Degree dist slope=-1.07  (< −1 → scale-free tendency)
  Top-5 hubs: [('ArthurPSmith', 1399), ('ديفيد عادل وهبة خليل', 1309), ('Pigsonthewing', 1252), ('Jura1', 1184), ('ChristianKl', 1113)]
  Dataset: PROPERTIES (MEDIUM)

  n=1,618  m=9,174  <k>=11.34  density=0.00701
  C=0.5592  L=2.9383  diameter=6
  C_rand=0.00701  L_rand=3.0428
  C/C_rand=79.74  L/L_rand=0.97  σ=82.57
  Degree dist slope=-1.24  (< −1 → scale-free tendency)
  Top-5 hubs: [('Jura1', 627), ('Infovarius', 241), ('VIGNERON', 219), ('Multichill', 193), ('Pasleim', 193)]
  Dataset: REQUEST_A_QUERY (SMALL)

  n=665  m=1,881  <k>=5.66  density=0.00852
  C=0.5383  L=2.5035  diamete